# Real-time Audio Processing
This notebook covers the four projects in this folder's README:
1. **Buffering and latency**: the callback model, latency, deadlines and dropouts (sections 1 and 3).
2. **Real-time effects**: processors that run block by block and keep their state between blocks (section 2), and real-time convolution reverb (section 5).
3. **Audio streaming**: ring buffers and jitter buffers for audio that arrives unevenly (section 4).
4. **Real-time synthesis**: oscillators and envelopes that stay continuous across blocks (section 6).

Most machines running this notebook (and all containers) have no audio interface, so sections 1 to 6 simulate the audio driver: its clock, its buffers and its deadlines. Section 7 runs the same processors on a real audio interface when one is available.

Python is fine for learning these ideas and for many real applications, but it cannot guarantee timing: the interpreter and its garbage collector can pause at any moment. Commercial plug-ins and audio applications are written in C, C++ or Rust for that reason. The rules in section 3 apply in any language.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython import display
import ipywidgets as widgets
from ipywidgets import interactive
from audiodsp import multi_sine, rbj_biquad, sine, time_axis

%matplotlib inline
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (14, 5)

FS = 48000   # the usual default rate of audio interfaces
rng = np.random.default_rng(0)

def play(x, label):
    print(f'{label}:')
    display.display(display.Audio(np.clip(x, -1, 1), rate=FS, normalize=False))

# Test material: three seconds of a plucked melody
def pluck(frequency, duration):
    t = time_axis(duration, FS)
    note = sum(np.sin(2 * np.pi * k * frequency * t) / k for k in range(1, 7))
    return note * (1 - np.exp(-t / 0.003)) * np.exp(-t / 0.25)

music = np.zeros(3 * FS)
for i, f in enumerate([220, 277.2, 329.6, 440, 329.6, 277.2, 220, 165, 220, 277.2, 329.6, 440]):
    start = int(i * 0.25 * FS)
    music[start:start + int(0.25 * FS)] += pluck(f, 0.25)
music *= 0.4 / np.max(np.abs(music))

## 1. The callback model and latency
An audio interface runs on its own clock. Its driver collects input samples into **blocks** (also called buffers or periods) of $B$ samples, and every $B / f_s$ seconds calls your **callback** function with the newest input block. The callback must return an output block before the interface needs it. It is not called when *you* are ready; it is called when the hardware is.

Every buffer adds delay. The input block is only handed over once it is full, which takes $B / f_s$. The output then waits in a queue of buffers before the converter plays it. With $n$ buffers in total between input and output, the round-trip **latency** is

$$\text{latency} = n \cdot \frac{B}{f_s} \;+\; \text{converter delay (typically around 1 ms)}$$

Small blocks give low latency but more callbacks per second, each with a fixed overhead, and less time to absorb a slow callback. Musicians playing through a computer notice delays above roughly 10 ms. Playback-only applications such as media players can use large buffers and hundreds of milliseconds of latency.

In [ ]:
print(f'{"Block":>6}{"Duration":>11}{"Callbacks/s":>13}{"Latency, 2 buffers":>21}{"Latency, 3 buffers":>21}')
for B in [32, 64, 128, 256, 512, 1024, 2048]:
    duration_ms = 1000 * B / FS
    print(f'{B:>6}{duration_ms:>8.2f} ms{FS / B:>13.0f}{2 * duration_ms:>18.1f} ms{3 * duration_ms:>18.1f} ms')

## 2. Block processors
A real-time effect sees one block at a time and never the whole signal. Anything it needs from the past, filter state, delayed samples, the current gain, it must keep itself between calls. The test for a correct block processor: its output must be identical to processing the whole signal at once, whatever the block size. `03_Digital_Filters/digital_filters.ipynb` section 2 showed what goes wrong when filter state is not kept.

Each processor below has a `process(block)` method that returns the output block.
- **`Biquads`**: a cascade of second-order sections that carries its filter state (`zi`) from block to block.
- **`FeedbackDelay`**: an echo that repeats and fades, $y[n] = x[n] + g\,y[n-D]$. Its memory is a **circular buffer**: a fixed array holding the last $D$ outputs, with a position that wraps around, so nothing is ever shifted or reallocated. Requiring $D \ge B$ means every past output a block needs was finished in an earlier block, so a whole block can be computed at once.
- **`SmoothedGain`**: a volume control. Jumping straight to a new gain makes a step in the waveform, heard as a click. Repeated jumps, for example while a slider moves, sound like a buzz called **zipper noise**. Ramping from the old gain to the new one across a block avoids it.

In [ ]:
class Biquads:
    """Cascade of biquads (second-order sections) that keeps its state between blocks."""
    def __init__(self, sos):
        self.sos = np.asarray(sos)
        self.zi = np.zeros((len(self.sos), 2))

    def process(self, block):
        y, self.zi = signal.sosfilt(self.sos, block, zi=self.zi)
        return y

class FeedbackDelay:
    """y[n] = x[n] + feedback * y[n - delay], with the last `delay` outputs in a circular buffer."""
    def __init__(self, delay, feedback):
        self.delay, self.feedback = delay, feedback
        self.buffer = np.zeros(delay)
        self.pos = 0                     # buffer[pos] holds y[n - delay] for the next n

    def process(self, block):
        assert len(block) <= self.delay, 'blocks must not be longer than the delay'
        idx = (self.pos + np.arange(len(block))) % self.delay
        y = block + self.feedback * self.buffer[idx]
        self.buffer[idx] = y             # y[n] is needed again at n + delay
        self.pos = (self.pos + len(block)) % self.delay
        return y

class SmoothedGain:
    """Gain whose changes are ramped linearly across one block."""
    def __init__(self, gain=1.0, smooth=True):
        self.current = self.target = gain
        self.smooth = smooth

    def process(self, block):
        if not self.smooth:
            self.current = self.target
            return block * self.current
        ramp = np.linspace(self.current, self.target, len(block), endpoint=False)
        self.current = self.target
        return block * ramp

def run_blocks(processors, x, block_size):
    """Feed x through a chain of processors one block at a time, as a driver would."""
    out = np.empty_like(x)
    for start in range(0, len(x), block_size):
        block = x[start:start + block_size]
        for p in processors:
            block = p.process(block)
        out[start:start + len(block)] = block
    return out

def make_chain():
    eq = [np.concatenate(rbj_biquad('lowshelf', 200, FS, gain_db=-6)),
          np.concatenate(rbj_biquad('peaking', 2500, FS, q=1.0, gain_db=4))]
    return [Biquads(eq), FeedbackDelay(int(0.3 * FS), 0.4), SmoothedGain(0.8)]

# Offline reference: the same chain applied to the whole signal at once
eq_sos = make_chain()[0].sos
D = int(0.3 * FS)
a_delay = np.zeros(D + 1)
a_delay[0], a_delay[D] = 1.0, -0.4
offline = 0.8 * signal.lfilter([1.0], a_delay, signal.sosfilt(eq_sos, music))

for B in [64, 256, 1000, 4800]:          # 1000 does not divide the signal length, so the last block is short
    blocked = run_blocks(make_chain(), music, B)
    print(f'Block size {B:>4}: max difference from offline processing {np.max(np.abs(blocked - offline)):.1e}')
play(offline, 'EQ + feedback delay')

### Zipper noise
Below, a tone's gain is switched between 1.0 and 0.25 every 30 ms, as if someone were moving a volume control quickly. Switching instantly at block boundaries clicks at every change. Ramping across each block (here 256 samples, 5.3 ms) does not.

In [ ]:
tone = 0.5 * sine(330, 2.0, FS)
B = 256
outputs = {}
for smooth in (False, True):
    gain = SmoothedGain(1.0, smooth=smooth)
    out = np.empty_like(tone)
    for start in range(0, len(tone), B):
        gain.target = 1.0 if (start / FS) % 0.06 < 0.03 else 0.25
        out[start:start + B] = gain.process(tone[start:start + B])
    outputs[smooth] = out

view = slice(int(0.025 * FS), int(0.045 * FS))
t_ms = np.arange(len(tone))[view] / FS * 1000
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_ms, outputs[False][view], label='instant gain change')
ax.plot(t_ms, outputs[True][view], label='ramped over one block')
ax.set_xlabel('Time (ms)')
ax.set_title('A gain change from 1.0 to 0.25')
ax.legend(loc='upper right')
plt.show()
play(outputs[False], 'Instant changes (zipper noise)')
play(outputs[True], 'Ramped changes')

## 3. Deadlines and dropouts
If a callback is late, the interface has nothing to play. It plays silence or repeats old data, and you hear a click or a gap. This is a **dropout**, also called an **xrun** (an underrun on output, an overrun on input). A stream must meet its deadline every time. Being fast on average is not enough.

First, the real cost of the effect chain above, measured on this machine for different block sizes. Each call has a fixed overhead, so small blocks use a larger share of the time available (the **CPU load**).

In [ ]:
block_sizes = [32, 64, 128, 256, 512, 1024]
noise = rng.normal(0, 0.1, 5 * FS)
medians = []
print(f'{"Block":>6}{"Budget":>11}{"Median":>11}{"99.9th pct":>13}{"Max":>11}{"CPU load":>10}')
for B in block_sizes:
    chain = make_chain()
    times = []
    for start in range(0, len(noise) - B, B):
        t0 = time.perf_counter()
        block = noise[start:start + B]
        for p in chain:
            block = p.process(block)
        times.append(time.perf_counter() - t0)
    times = np.array(times) * 1e6
    budget = 1e6 * B / FS
    medians.append(np.median(times))
    print(f'{B:>6}{budget:>8.0f} µs{np.median(times):>8.0f} µs{np.percentile(times, 99.9):>10.0f} µs'
          f'{times.max():>8.0f} µs{100 * np.median(times) / budget:>9.1f}%')

per_sample_us, overhead_us = np.polyfit(block_sizes, medians, 1)
print(f'\nCost model fitted to the medians: {overhead_us:.0f} µs per call + {1000 * per_sample_us:.0f} ns per sample')

On average the chain uses a small fraction of its budget. Dropouts come from the rare slow calls: the operating system running something else, Python's garbage collector, memory allocation, a page fault. These pauses are hard to reproduce on demand, so the simulation below adds them deliberately.

**Simulated device.** Input block $k$ is complete at time $(k+1)T$, with $T = B/f_s$. The callback starts then, or when the previous callback finishes if that is later. With $n$ buffers, its output is needed $(n-1)T$ after the input was ready. Each callback costs what the fit above measured, varied randomly by about ±30%, and with a small probability it is held up by a pause of several milliseconds. A block that misses its deadline is played as silence.

More buffers give each callback more slack to recover from a pause, at the cost of latency. Try a pause probability of 0.5% (one in 200 callbacks) with a 3 ms pause, then increase the buffers.

The output's latency is measured the same way phase was measured in `01_Discrete_Time_Signals_and_Systems`: the lag that maximizes the cross-correlation between input and output.

In [ ]:
def simulate_device(x, block_size, n_buffers, pause_prob, pause_ms, seed=1, chain_factory=make_chain):
    """Stream x through a processing chain on a simulated device. Returns (output, number of dropouts)."""
    sim_rng = np.random.default_rng(seed)
    T = block_size / FS
    chain = chain_factory()
    out = np.zeros(len(x) + n_buffers * block_size)
    previous_finish, dropouts = 0.0, 0
    for k, start in enumerate(range(0, len(x), block_size)):
        ready = (k + 1) * T
        cost = 1e-6 * (overhead_us + per_sample_us * block_size) * sim_rng.lognormal(0, 0.3)
        if sim_rng.random() < pause_prob:
            cost += pause_ms / 1000
        finish = max(ready, previous_finish) + cost
        previous_finish = finish
        block = x[start:start + block_size]
        for p in chain:
            block = p.process(block)              # the processing still happens, and its state advances
        if finish <= ready + (n_buffers - 1) * T:
            out[start + n_buffers * block_size:start + n_buffers * block_size + len(block)] = block
        else:
            dropouts += 1                         # too late: this block is played as silence
    return out, dropouts

def measured_latency(x, y):
    lags = signal.correlation_lags(len(y), len(x), mode='full')
    r = signal.correlate(y, x, mode='full', method='fft')
    return lags[np.argmax(r)]

def device_tool(block_size=128, n_buffers=2, pause_prob=0.005, pause_ms=3.0):
    y, dropouts = simulate_device(music, block_size, n_buffers, pause_prob, pause_ms)
    # Measure the device's own latency with an empty chain: the EQ's phase shift would bias the peak by a few samples
    passthrough, _ = simulate_device(music, block_size, n_buffers, 0.0, 0.0, chain_factory=list)
    lag = measured_latency(music, passthrough)
    n_blocks = int(np.ceil(len(music) / block_size))
    fig, ax = plt.subplots(figsize=(14, 3.5))
    ax.plot(np.arange(len(y)) / FS, y, linewidth=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_title(f'{dropouts} dropouts in {n_blocks} blocks ({60 * dropouts / (len(music) / FS):.0f} per minute)')
    plt.show()
    print(f'Latency: {n_buffers} x {block_size} = {n_buffers * block_size} samples = {1000 * n_buffers * block_size / FS:.1f} ms. '
          f'Measured by cross-correlation: {lag} samples.')
    play(y, 'Output')

display.display(interactive(
    device_tool,
    block_size=widgets.Dropdown(options=[32, 64, 128, 256, 512, 1024], value=128, description='Block:'),
    n_buffers=widgets.IntSlider(value=2, min=2, max=6, description='Buffers:'),
    pause_prob=widgets.FloatLogSlider(value=0.005, base=10, min=-4, max=-1, step=0.25, readout_format='.4f', description='Pause prob.:'),
    pause_ms=widgets.FloatSlider(value=3.0, min=0.5, max=20.0, step=0.5, description='Pause (ms):'),
))

**Rules for code that runs in an audio callback**, in any language. Each one removes a source of unpredictable delay:
- Do not allocate or free memory. Create buffers once, up front.
- Do not wait on locks, files, the network or the screen. Pass data to and from other threads through lock-free queues and ring buffers (next section).
- Do not print or log. Count problems and report them from another thread.
- Keep the cost of each call predictable. An algorithm that is usually fast but occasionally slow will drop out.

(The Python code in this notebook allocates new arrays in every call. That is acceptable here, but is one reason Python cannot promise to meet deadlines.)

## 4. Streaming and ring buffers
When audio arrives from a network, a file being downloaded, or another thread, it does not arrive at the steady pace the audio callback consumes it. A **ring buffer** (circular FIFO) sits between the two. The producer writes whatever has arrived, the callback reads exactly one block per call, and neither ever waits for the other.

Before playback starts, the buffer is **prefilled** with some audio. That reserve covers late arrivals: if packets are delayed by more than the reserve, the callback finds the buffer empty and plays silence. A larger prefill means fewer dropouts but more latency. A buffer used this way is called a **jitter buffer**. Every internet call and streaming service has one.

The simulation sends 20 ms packets, the usual size for internet voice calls. Each takes 10 ms plus a random extra delay to arrive (typically a few milliseconds), and 1% of packets are held up by an extra 80 ms. Packets arrive in order: a delayed packet also delays the ones behind it. Because audio arrives 20 ms at a time, any prefill up to 20 ms behaves the same: playback starts when the first packet arrives.

In [ ]:
class RingBuffer:
    """Fixed-capacity FIFO for audio samples. Never blocks: callers check what was written or read."""
    def __init__(self, capacity):
        self.data = np.zeros(capacity)
        self.capacity, self.read_pos, self.count = capacity, 0, 0

    def write(self, samples):
        n = min(len(samples), self.capacity - self.count)          # on overflow, the excess is dropped
        idx = (self.read_pos + self.count + np.arange(n)) % self.capacity
        self.data[idx] = samples[:n]
        self.count += n
        return n

    def read(self, n):
        k = min(n, self.count)                                     # on underrun, the rest is silence
        out = np.zeros(n)
        out[:k] = self.data[(self.read_pos + np.arange(k)) % self.capacity]
        self.read_pos = (self.read_pos + k) % self.capacity
        self.count -= k
        return out, k

def simulate_stream(x, prefill_ms, block_size=256, packet_ms=20, seed=2):
    """Packets arrive with network jitter; the callback reads one block every block period."""
    sim_rng = np.random.default_rng(seed)
    packet = int(packet_ms / 1000 * FS)
    n_packets = int(np.ceil(len(x) / packet))
    delays = 0.010 + sim_rng.gamma(2.0, 0.002, n_packets) + 0.080 * (sim_rng.random(n_packets) < 0.01)
    arrivals = np.maximum.accumulate(np.arange(n_packets) * packet_ms / 1000 + delays)   # in-order delivery

    ring = RingBuffer(FS)                                          # one second of capacity
    prefill = int(prefill_ms / 1000 * FS)
    received = np.cumsum(np.minimum(packet, len(x) - np.arange(n_packets) * packet))
    start_time = arrivals[np.searchsorted(received, min(prefill, len(x)))]   # playback starts once prefilled

    period = block_size / FS
    out, fill, read_times = [], [], []
    next_packet, underruns = 0, 0
    for k in range(int(np.ceil(len(x) / block_size))):
        now = start_time + k * period
        while next_packet < n_packets and arrivals[next_packet] <= now:
            ring.write(x[next_packet * packet:(next_packet + 1) * packet])
            next_packet += 1
        block, got = ring.read(block_size)
        underruns += got < block_size
        out.append(block)
        fill.append(ring.count / FS * 1000)
        read_times.append(now)
    latency_ms = 1000 * start_time       # the first sample was sent at time 0
    return np.concatenate(out)[:len(x)], np.array(read_times), np.array(fill), underruns, latency_ms

def stream_tool(prefill_ms=40):
    y, read_times, fill, underruns, latency_ms = simulate_stream(music, prefill_ms)
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].plot(read_times, fill)
    axes[0].axhline(0, color='r', linewidth=1)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Buffered audio (ms)')
    axes[0].set_title(f'Jitter buffer fill level, prefill {prefill_ms} ms')
    axes[1].plot(np.arange(len(y)) / FS, y, linewidth=0.5)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_title(f'{underruns} underrun blocks, latency {latency_ms:.0f} ms')
    plt.tight_layout()
    plt.show()
    play(y, 'Received audio')

display.display(interactive(
    stream_tool,
    prefill_ms=widgets.IntSlider(value=40, min=0, max=200, step=10, description='Prefill (ms):'),
))

In [ ]:
print(f'{"Prefill":>8}{"Latency":>10}{"Underrun blocks":>17}   (averaged over 20 random networks, 3 s each)')
for prefill_ms in [0, 20, 40, 60, 80, 100, 150]:
    results = [simulate_stream(music, prefill_ms, seed=s) for s in range(20)]
    print(f'{prefill_ms:>5} ms{np.mean([r[4] for r in results]):>7.0f} ms{np.mean([r[3] for r in results]):>17.1f}')

## 5. Real-time convolution reverb
`05_Audio_Effects` applies a reverb by convolving a whole recording with a room impulse response (RIR) in one call. In real time, each block must be convolved as it arrives, and the tail of every earlier block's response added in. Two ways to do that:
- **Overlap-add with the whole RIR**: every block, take an FFT big enough for the block plus the whole RIR. A 2 s RIR at 48 kHz needs a 131,072-point FFT and its inverse for every block, even a 64-sample one.
- **Uniformly partitioned convolution**: cut the RIR into pieces one block long and take the FFT of each piece once, up front. Each new input block is transformed once (a $2B$-point FFT) and kept in a **frequency-domain delay line**. The output block is the sum, over all pieces, of each piece's spectrum times the input spectrum from that many blocks ago, followed by one inverse FFT. Per block that is two small FFTs plus about as many complex multiplications as the RIR has samples.

Neither adds latency beyond the block itself, and both must give exactly the same output as offline convolution. Both still touch the whole impulse response for every block, so their cost per block is roughly the same whatever the block size, and their share of the budget grows as blocks get smaller.

In [ ]:
class OverlapAddConvolver:
    """Each block: one FFT large enough for block + whole impulse response, then overlap-add the tail."""
    def __init__(self, h, block_size):
        self.n_fft = int(2 ** np.ceil(np.log2(block_size + len(h) - 1)))
        self.H = np.fft.rfft(h, self.n_fft)
        self.tail = np.zeros(self.n_fft)

    def process(self, block):
        y = np.fft.irfft(np.fft.rfft(block, self.n_fft) * self.H, self.n_fft) + self.tail
        n = len(block)
        self.tail = np.concatenate((y[n:], np.zeros(n)))
        return y[:n]

class PartitionedConvolver:
    """Uniformly partitioned overlap-save convolution with a frequency-domain delay line."""
    def __init__(self, h, block_size):
        B = self.B = block_size
        n_parts = -(-len(h) // B)
        h = np.concatenate((h, np.zeros(n_parts * B - len(h))))
        self.H = np.fft.rfft(h.reshape(n_parts, B), 2 * B, axis=1)   # one spectrum per piece
        self.fdl = np.zeros_like(self.H)                              # past input spectra, circular
        self.pos = 0
        self.window = np.zeros(2 * B)                                 # previous block + current block

    def process(self, block):
        B, n = self.B, len(block)
        self.window[:B] = self.window[B:]
        self.window[B:] = 0
        self.window[B:B + n] = block
        self.fdl[self.pos] = np.fft.rfft(self.window)
        newest_first = (self.pos - np.arange(len(self.H))) % len(self.H)
        Y = np.einsum('pk,pk->k', self.fdl[newest_first], self.H)
        self.pos = (self.pos + 1) % len(self.H)
        return np.fft.irfft(Y, 2 * B)[B:B + n]     # overlap-save: the second half is valid

# A 2 s synthetic room impulse response: direct sound plus a tail decaying 60 dB in 1.6 s
t_h = time_axis(2.0, FS)
rir = 0.5 * rng.normal(0, 1, len(t_h)) * np.exp(-t_h * np.log(1000) / 1.6) / np.sqrt(FS * 1.6 / 13.8)
rir[0] = 1.0
reference = signal.fftconvolve(music, rir)[:len(music)]

print(f'Impulse response: {len(rir)} samples. Output compared with offline fftconvolve; time per block vs budget.')
print(f'{"Block":>6}{"Method":>14}{"Max diff":>11}{"Median time":>14}{"Budget":>10}{"Load":>8}')
for B in [32, 64, 256, 1024]:
    for name, cls in [('overlap-add', OverlapAddConvolver), ('partitioned', PartitionedConvolver)]:
        conv = cls(rir, B)
        out, times = np.empty_like(music), []
        for start in range(0, len(music), B):
            t0 = time.perf_counter()
            out[start:start + B] = conv.process(music[start:start + B])
            times.append(time.perf_counter() - t0)
        budget = B / FS
        print(f'{B:>6}{name:>14}{np.max(np.abs(out - reference)):>11.1e}{1e3 * np.median(times):>11.2f} ms'
              f'{1e3 * budget:>7.2f} ms{100 * np.median(times) / budget:>7.0f}%')
play(0.6 * music + 0.4 * reference / np.max(np.abs(reference)) * np.max(np.abs(music)), 'Reverb (wet/dry mix)')

The median times stay nearly constant down the table, as expected, and the load rises as blocks shrink. Partitioned convolution is several times cheaper per block, which decides whether small blocks fit in the budget at all. A load near or above 100% means continuous dropouts. Commercial convolution reverbs go further with **non-uniform** partitions: small pieces at the start of the response, which keep the latency low, and larger pieces for the long tail, which need far fewer multiplications in total.

## 6. Real-time synthesis
A synthesizer voice generates audio block by block, so its oscillator and envelope must continue smoothly from one block to the next.
- **Oscillator.** Computing each block as $\sin(2\pi f n / f_s)$ with $n$ restarting at 0 resets the phase at every block boundary. Unless the block happens to hold a whole number of cycles, the waveform jumps, which buzzes at the block rate. A **phase accumulator** keeps the running phase between blocks and adds $2\pi f / f_s$ per sample. Changing frequency then changes how fast the phase advances, never where it is, so pitch changes are seamless too.
- **Envelope.** Starting or stopping a note at full level clicks. A short attack and release ramp, carried across blocks, avoids it.

In this simulation, note events take effect at block boundaries, so their timing is quantized to one block (5.3 ms at 256 samples). Real synthesizers apply each event at its exact sample offset within the block.

In [ ]:
class SineVoice:
    """Sine oscillator with a phase accumulator and a linear attack/release envelope."""
    def __init__(self, attack=0.005, release=0.05, continuous=True):
        self.phase, self.freq, self.level, self.target = 0.0, 440.0, 0.0, 0.0
        self.attack_step, self.release_step = 1 / (attack * FS), 1 / (release * FS)
        self.continuous = continuous

    def note_on(self, freq):
        self.freq, self.target = freq, 1.0

    def note_off(self):
        self.target = 0.0

    def render(self, n):
        increment = 2 * np.pi * self.freq / FS
        if self.continuous:
            phases = self.phase + increment * np.arange(n)
            self.phase = (phases[-1] + increment) % (2 * np.pi)
            steps = np.arange(1, n + 1)
            if self.target > self.level:
                envelope = np.minimum(self.level + self.attack_step * steps, self.target)
            else:
                envelope = np.maximum(self.level - self.release_step * steps, self.target)
        else:   # restart the phase every block and switch the level instantly
            phases = increment * np.arange(n)
            envelope = np.full(n, self.target)
        self.level = envelope[-1]
        return 0.3 * envelope * np.sin(phases)

# (time in seconds, MIDI note or None for note off)
events = [(0.0, 60), (0.4, 64), (0.8, 67), (1.2, None), (1.4, 72), (1.8, 71), (2.0, 69), (2.2, 67), (2.6, None)]
B = 256
renders = {}
for continuous in (False, True):
    voice = SineVoice(continuous=continuous)
    out = np.zeros(3 * FS)
    pending = list(events)
    for start in range(0, len(out), B):
        while pending and pending[0][0] * FS <= start:
            _, note = pending.pop(0)
            if note is None:
                voice.note_off()
            else:
                voice.note_on(440 * 2 ** ((note - 69) / 12))
        out[start:start + B] = voice.render(min(B, len(out) - start))
    renders[continuous] = out

view = slice(int(0.39 * FS), int(0.41 * FS))
t_ms = np.arange(len(out))[view] / FS * 1000
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].plot(t_ms, renders[False][view], label='phase restarts every block')
axes[0].plot(t_ms, renders[True][view], label='phase accumulator')
for boundary in range(-(-view.start // B) * B, view.stop, B):
    axes[0].axvline(boundary / FS * 1000, color='0.7', linewidth=0.8)
axes[0].set_xlabel('Time (ms)   (grey lines: block boundaries)')
axes[0].set_title('Around the change from C to E at 0.4 s')
axes[0].legend(loc='lower left')
for continuous, label in [(False, 'restarting phase'), (True, 'phase accumulator')]:
    f, mag = signal.welch(renders[continuous], FS, nperseg=8192)
    axes[1].semilogy(f, mag, label=label)
axes[1].set_xlim(0, 4000)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_title('Spectrum: the restarting oscillator spreads energy everywhere')
axes[1].legend()
plt.tight_layout()
plt.show()
play(renders[False], 'Phase restarts every block, no envelope')
play(renders[True], 'Phase accumulator and envelope')

## 7. Live audio (optional)
The processors above are written in the form an audio driver expects, so they run unchanged on a real audio interface. This cell uses the [`sounddevice`](https://python-sounddevice.readthedocs.io) package, which needs the PortAudio library:
```
pip install sounddevice
sudo apt install libportaudio2      # Linux only; macOS and Windows wheels include PortAudio
```
It processes the default input (a microphone) through the effect chain from section 2 and plays the result for 10 seconds. **Use headphones**, or the speakers will feed back into the microphone. Set `RUN_LIVE = True` to run it. Containers and remote machines usually have no audio device.

`sounddevice` reports dropouts through the `status` argument, and the latency it actually achieved through `stream.latency`.

In [ ]:
RUN_LIVE = False

try:
    import sounddevice as sd
    sd.query_devices()
except Exception as e:          # ImportError, or OSError when the PortAudio library is missing
    sd = None
    print(f'Live audio not available here: {e}')

if sd is not None and RUN_LIVE:
    chain = make_chain()
    problems = []

    def callback(indata, outdata, frames, time_info, status):
        if status:
            problems.append(str(status))       # do not print inside the callback
        block = indata[:, 0].astype(np.float64)
        for p in chain:
            block = p.process(block)
        outdata[:, 0] = block

    with sd.Stream(samplerate=FS, blocksize=256, channels=1, callback=callback, latency='low') as stream:
        print(f'Running for 10 s. Input + output latency reported by the driver: {1000 * sum(stream.latency):.1f} ms')
        sd.sleep(10_000)
    print(f'Dropouts and other problems reported: {len(problems)}', problems[:5])